<a href="https://colab.research.google.com/github/snigdhasudheer/BirdVision/blob/main/TerraSpectra_Day04_Spectral_Data_Preprocessing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# TerraSpectra — Day 4: Hyperspectral Data Cleaning & Preprocessing

**Official Day 4 task (AI/ML):** clean the hyperspectral data, inspect invalid values, handle missing/non-finite values, and implement a reproducible normalization/preprocessing pipeline.

**Input from earlier work**
- Day 2: `hyperspectral_normalized.npy` — expected shape `(512, 512, 204)`.
- Day 3 (optional): `annotated_plant_spectra.csv` — 244 annotated coordinates with 204 spectral-band columns, plus `plant_annotations.csv` and `plant_annotation_mask.npy`.

> Important: Day 2 already produced a standardized cube (mean approximately 0 and standard deviation approximately 1). This notebook verifies that result and applies a controlled, per-band preprocessing step. It does not blindly normalize the same array multiple times. Keep the Day 2 file unchanged.

Run all cells from top to bottom in Google Colab. Upload the input files when prompted.

In [ ]:
# 1. Import libraries and set reproducibility options
import os
import zipfile
import shutil
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from google.colab import files

np.random.seed(42)

print("TerraSpectra Day 4 - libraries imported successfully!")
print("NumPy:", np.__version__)
print("Pandas:", pd.__version__)

## 1. Upload the Day 2 and Day 3 inputs

Required:
- `hyperspectral_normalized.npy`

Optional Day 3 files:
- `annotated_plant_spectra.csv`
- `plant_annotations.csv`
- `plant_annotation_mask.npy`

You can upload the Day 3 ZIP (`TerraSpectra_Day03_outputs.zip`) instead of the individual optional files. The cell extracts it automatically.

In [ ]:
# 2. Upload inputs
required_cube = "/content/hyperspectral_normalized.npy"
optional_files = [
    "/content/annotated_plant_spectra.csv",
    "/content/plant_annotations.csv",
    "/content/plant_annotation_mask.npy"
]

if not os.path.exists(required_cube):
    print("Upload hyperspectral_normalized.npy and, if available, the Day 3 output ZIP/files.")
    uploaded = files.upload()
    print("Uploaded:", list(uploaded.keys()))

# Extract a Day 3 ZIP if one was uploaded
for name in os.listdir("/content"):
    if name.lower().endswith(".zip") and "day03" in name.lower():
        with zipfile.ZipFile(os.path.join("/content", name), "r") as zf:
            zf.extractall("/content/TerraSpectra_Day03_input")
        extracted_root = "/content/TerraSpectra_Day03_input"
        for root, dirs, filenames in os.walk(extracted_root):
            for filename in filenames:
                src = os.path.join(root, filename)
                dst = os.path.join("/content", filename)
                if not os.path.exists(dst):
                    shutil.copy2(src, dst)
        print("Extracted Day 3 ZIP:", name)

if not os.path.exists(required_cube):
    raise FileNotFoundError(
        "hyperspectral_normalized.npy is required. Save it from Day 2 and upload it here."
    )

print("Required Day 2 cube found.")
print("Optional Day 3 files found:", [os.path.basename(p) for p in optional_files if os.path.exists(p)])

## 2. Load and inspect the hyperspectral cube

Expected dimensions are height × width × spectral bands = `(512, 512, 204)`. We inspect shape, dtype, finite values, and basic statistics before changing anything.

In [ ]:
# 3. Load hyperspectral cube
cube = np.load(required_cube)

print("Cube shape:", cube.shape)
print("Data type:", cube.dtype)
print("Minimum:", float(np.nanmin(cube)))
print("Maximum:", float(np.nanmax(cube)))
print("Mean:", float(np.nanmean(cube)))
print("Standard deviation:", float(np.nanstd(cube)))
print("NaN count:", int(np.isnan(cube).sum()))
print("Positive infinity count:", int(np.isposinf(cube).sum()))
print("Negative infinity count:", int(np.isneginf(cube).sum()))
print("Zero count:", int(np.count_nonzero(cube == 0)))

assert cube.ndim == 3, "Expected a 3D hyperspectral cube."
assert cube.shape[-1] == 204, "Expected 204 spectral bands. Check the uploaded file."
assert cube.shape[:2] == (512, 512), "Expected spatial dimensions of 512 x 512."

print("\nInput shape verification successful.")

## 3. Inspect invalid values band by band

NaN and positive/negative infinity are invalid numeric values. Zero values are counted separately and are **not** automatically treated as missing, because zero may represent valid background or sensor data.

For any band containing invalid values, this pipeline replaces them with that band's median calculated from finite pixels. If an entire band has no finite values, the notebook stops rather than inventing data.

In [ ]:
# 4. Create a band-wise quality report
band_report = []
for band_idx in range(cube.shape[2]):
    band = cube[:, :, band_idx]
    finite = np.isfinite(band)
    vals = band[finite]
    band_report.append({
        "band": band_idx + 1,
        "finite_count": int(finite.sum()),
        "invalid_count": int((~finite).sum()),
        "zero_count": int(np.count_nonzero(band == 0)),
        "minimum": float(vals.min()) if vals.size else np.nan,
        "maximum": float(vals.max()) if vals.size else np.nan,
        "mean": float(vals.mean()) if vals.size else np.nan,
        "std": float(vals.std()) if vals.size else np.nan
    })

band_quality = pd.DataFrame(band_report)
display(band_quality.head(10))
print("Total invalid values:", int((~np.isfinite(cube)).sum()))
print("Bands with invalid values:", int((band_quality["invalid_count"] > 0).sum()))
print("Bands with no finite values:", int((band_quality["finite_count"] == 0).sum()))

In [ ]:
# 5. Visualize invalid values per spectral band
plt.figure(figsize=(12, 4))
plt.plot(band_quality["band"], band_quality["invalid_count"], marker=".")
plt.title("TerraSpectra - Invalid Values per Spectral Band")
plt.xlabel("Spectral Band")
plt.ylabel("NaN / Infinite Values")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 4. Handle missing and non-finite spectral values

The cube is copied before cleaning. Each affected band uses its finite-value median as the replacement value. The original Day 2 array remains unchanged.

In [ ]:
# 6. Replace NaN and +/- infinity using per-band medians
clean_cube = cube.astype(np.float32, copy=True)
replacement_counts = []

for band_idx in range(clean_cube.shape[2]):
    band = clean_cube[:, :, band_idx]
    finite_mask = np.isfinite(band)

    if not finite_mask.any():
        raise ValueError(f"Band {band_idx + 1} contains no finite values; cannot safely impute it.")

    invalid_mask = ~finite_mask
    count = int(invalid_mask.sum())

    if count:
        band_median = float(np.median(band[finite_mask]))
        band[invalid_mask] = band_median

    replacement_counts.append({
        "band": band_idx + 1,
        "replaced_values": count
    })

replacement_report = pd.DataFrame(replacement_counts)
print("Values replaced:", int(replacement_report["replaced_values"].sum()))
print("Remaining invalid values:", int((~np.isfinite(clean_cube)).sum()))
display(replacement_report[replacement_report["replaced_values"] > 0].head(20))

assert np.isfinite(clean_cube).all(), "Cleaning failed: invalid values remain."
print("Missing/non-finite value handling successful.")

## 5. Normalize spectral bands safely

The Day 2 cube is already globally standardized. Here we calculate per-band statistics and standardize each band independently (z-score). Constant bands are assigned zero after centering to avoid division by zero.

This produces a reproducible per-band normalized cube for downstream PCA/model preparation. The output is saved separately as `hyperspectral_day04_preprocessed.npy`.

In [ ]:
# 7. Per-band z-score normalization
preprocessed_cube = np.empty_like(clean_cube, dtype=np.float32)
normalization_rows = []

for band_idx in range(clean_cube.shape[2]):
    band = clean_cube[:, :, band_idx]
    band_mean = float(np.mean(band))
    band_std = float(np.std(band))

    if band_std > 1e-8:
        preprocessed_cube[:, :, band_idx] = (band - band_mean) / band_std
    else:
        preprocessed_cube[:, :, band_idx] = 0.0

    normalization_rows.append({
        "band": band_idx + 1,
        "input_mean": band_mean,
        "input_std": band_std,
        "output_mean": float(preprocessed_cube[:, :, band_idx].mean()),
        "output_std": float(preprocessed_cube[:, :, band_idx].std()),
        "constant_band": bool(band_std <= 1e-8)
    })

normalization_report = pd.DataFrame(normalization_rows)
display(normalization_report.head(10))

print("Preprocessed shape:", preprocessed_cube.shape)
print("Preprocessed dtype:", preprocessed_cube.dtype)
print("Overall mean:", float(preprocessed_cube.mean()))
print("Overall standard deviation:", float(preprocessed_cube.std()))
print("All values finite:", bool(np.isfinite(preprocessed_cube).all()))

In [ ]:
# 8. Compare a sample band's distribution before and after preprocessing
sample_band = 100  # zero-based index; displays spectral band 101

plt.figure(figsize=(10, 5))
plt.hist(clean_cube[:, :, sample_band].ravel(), bins=60, alpha=0.55, label="Cleaned input")
plt.hist(preprocessed_cube[:, :, sample_band].ravel(), bins=60, alpha=0.55, label="Per-band normalized")
plt.title(f"TerraSpectra - Band {sample_band + 1} Distribution")
plt.xlabel("Pixel value")
plt.ylabel("Frequency")
plt.legend()
plt.grid(True, alpha=0.2)
plt.tight_layout()
plt.show()

## 6. Validate optional Day 3 annotated spectral table

If `annotated_plant_spectra.csv` is available, this section checks its dimensions and missing/non-finite spectral values, then creates a cleaned copy. It does not change the annotation coordinates or claim that the annotated pixels are diseased; Day 3 labels them as annotated plant pixels.

In [ ]:
# 9. Clean Day 3 extracted spectral table if present
spectra_path = "/content/annotated_plant_spectra.csv"
clean_spectra_df = None
spectral_feature_columns = []

if os.path.exists(spectra_path):
    spectra_df = pd.read_csv(spectra_path)
    spectral_feature_columns = [c for c in spectra_df.columns if c.startswith("band_")]

    print("Day 3 spectral table shape:", spectra_df.shape)
    print("Spectral feature columns:", len(spectral_feature_columns))
    print("Missing values before cleaning:", int(spectra_df[spectral_feature_columns].isna().sum().sum()))

    if len(spectral_feature_columns) != 204:
        raise ValueError("Expected 204 band_* columns in annotated_plant_spectra.csv.")

    clean_spectra_df = spectra_df.copy()
    for col in spectral_feature_columns:
        values = pd.to_numeric(clean_spectra_df[col], errors="coerce")
        values = values.replace([np.inf, -np.inf], np.nan)
        median_value = values.median()
        if pd.isna(median_value):
            raise ValueError(f"{col} contains no valid values; cannot safely impute.")
        clean_spectra_df[col] = values.fillna(median_value)

    print("Missing/non-finite spectral values after cleaning:",
          int(clean_spectra_df[spectral_feature_columns].isna().sum().sum()))
    display(clean_spectra_df.head())
else:
    print("Optional Day 3 annotated_plant_spectra.csv not uploaded; skipping this section.")

## 7. Save Day 4 preprocessing outputs

Outputs include the cleaned cube, per-band quality report, normalization report, and (if supplied) cleaned annotated spectra. These are saved in a dedicated folder and packaged into a ZIP for download.

In [ ]:
# 10. Save Day 4 outputs
output_dir = "/content/TerraSpectra_Day04"
os.makedirs(output_dir, exist_ok=True)

np.save(os.path.join(output_dir, "hyperspectral_day04_preprocessed.npy"), preprocessed_cube)
band_quality.to_csv(os.path.join(output_dir, "band_quality_report.csv"), index=False)
replacement_report.to_csv(os.path.join(output_dir, "missing_value_replacement_report.csv"), index=False)
normalization_report.to_csv(os.path.join(output_dir, "per_band_normalization_report.csv"), index=False)

if clean_spectra_df is not None:
    clean_spectra_df.to_csv(
        os.path.join(output_dir, "annotated_plant_spectra_cleaned.csv"),
        index=False
    )

print("Day 4 files saved:")
for filename in sorted(os.listdir(output_dir)):
    print("-", filename)

In [ ]:
# 11. Package outputs into a ZIP archive
zip_path = shutil.make_archive(
    "/content/TerraSpectra_Day04_outputs",
    "zip",
    output_dir
)

print("Created:", zip_path)
files.download(zip_path)

## 8. Day 4 completion report

Check that the preprocessed cube has the expected dimensions, all values are finite, and every non-constant band is approximately zero-mean and unit-standard-deviation.

In [ ]:
# 12. Final verification
print("=" * 60)
print("TERRASPECTRA - DAY 4 COMPLETION REPORT")
print("=" * 60)
print("Original cube shape:", cube.shape)
print("Cleaned cube shape:", clean_cube.shape)
print("Preprocessed cube shape:", preprocessed_cube.shape)
print("Input invalid values:", int((~np.isfinite(cube)).sum()))
print("Replaced invalid values:", int(replacement_report["replaced_values"].sum()))
print("Remaining invalid values:", int((~np.isfinite(preprocessed_cube)).sum()))
print("Number of spectral bands:", preprocessed_cube.shape[2])
print("Preprocessed mean:", float(preprocessed_cube.mean()))
print("Preprocessed standard deviation:", float(preprocessed_cube.std()))
print("Output directory:", output_dir)

assert preprocessed_cube.shape == (512, 512, 204)
assert np.isfinite(preprocessed_cube).all()

print("\nDAY 4 PREPROCESSING COMPLETED SUCCESSFULLY.")